In [1]:
import sklearn
import os
import s3fs
import fsspec as fs
import pandas as pd

In [2]:
sklearn.__version__

os.environ['AWS_S3_ENDPOINT']

S3_ENDPOINT_URL = 'http://' +os.environ['AWS_S3_ENDPOINT']
S3_ENDPOINT_URL

fs = s3fs.S3FileSystem(client_kwargs = {'endpoint_url' : S3_ENDPOINT_URL})

fs.ls('') 

['mlepennec-ensae']

In [3]:
BUCKET = 'mlepennec-ensae'
FILE_KEY_S3 = '/readmission_avc.parquet'
FILE_PATH_S3 = BUCKET + FILE_KEY_S3

In [4]:
with fs.open(FILE_PATH_S3, mode='rb') as file_in :
    dataini = pd.read_parquet(file_in)

In [6]:
dataini

,modeEntree,modeSortie,duree,ghm2,dp,sexe,age,nbActe,nbRum,nbda,id,id_D
0,8,9,0,01M37E,I671,2.0,76.0,4,1,NaN,l19,
1,8,8,3,01C061,I652,2.0,77.0,4,1,1.0,s7e,
2,8,7,13,01M303,I634,NaN,NaN,4,1,7.0,23f,
3,8,8,11,01M301,I639,1.0,83.0,4,2,2.0,8oi,None
4,8,6,8,01M303,I635,1.0,71.0,4,1,9.0,otz,ld
...,...,...,...,...,...,...,...,...,...,...,...,...
1695,8,7,1,01M30T,I614,1.0,88.0,4,1,4.0,kjg,
1696,8,6,10,01M303,I635,1.0,81.0,10,3,7.0,gie,my
1697,8,8,8,01M301,I639,1.0,68.0,5,3,6.0,6bl,
1698,8,8,11,01M301,I676,2.0,28.0,16,5,7.0,7m8,


# 1. Data preprocessing

In [194]:
dataini.dtypes

modeEntree      int32
modeSortie      int32
duree           int32
ghm2           object
dp             object
sexe          float64
age           float64
nbActe          int32
nbRum           int32
nbda          float64
id             object
id_D           object
dtype: object

In [195]:
dataini.isna().sum()

modeEntree      0
modeSortie      0
duree           0
ghm2            0
dp              0
sexe           20
age            20
nbActe          0
nbRum           0
nbda          134
id              0
id_D          200
dtype: int64

In [196]:
dataini = dataini.dropna(axis=0, subset=['id_D']).copy()

In [197]:
dataini['rea'] = (dataini['id_D'] != '').astype('int8')
dataini

,modeEntree,modeSortie,duree,ghm2,dp,sexe,age,nbActe,nbRum,nbda,id,id_D,rea
0,8,9,0,01M37E,I671,2.0,76.0,4,1,NaN,l19,,0
1,8,8,3,01C061,I652,2.0,77.0,4,1,1.0,s7e,,0
2,8,7,13,01M303,I634,NaN,NaN,4,1,7.0,23f,,0
4,8,6,8,01M303,I635,1.0,71.0,4,1,9.0,otz,ld,1
6,8,7,8,01M302,I639,1.0,92.0,7,2,4.0,np7,,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1695,8,7,1,01M30T,I614,1.0,88.0,4,1,4.0,kjg,,0
1696,8,6,10,01M303,I635,1.0,81.0,10,3,7.0,gie,my,1
1697,8,8,8,01M301,I639,1.0,68.0,5,3,6.0,6bl,,0
1698,8,8,11,01M301,I676,2.0,28.0,16,5,7.0,7m8,,0


In [198]:
dataini.rea.value_counts()

rea
0    1281
1     219
Name: count, dtype: int64

In [199]:
str_cols = ['modeEntree', 'modeSortie', 'sexe']
dataini[str_cols] = dataini[str_cols].astype("object")

In [200]:
dataini.isna().sum()

modeEntree      0
modeSortie      0
duree           0
ghm2            0
dp              0
sexe           19
age            19
nbActe          0
nbRum           0
nbda          121
id              0
id_D            0
rea             0
dtype: int64

In [201]:
dataini.nbda.value_counts()

nbda
3.0     193
4.0     178
2.0     154
1.0     150
5.0     135
6.0     129
7.0     115
8.0      85
9.0      68
10.0     37
11.0     32
13.0     27
12.0     20
14.0     14
15.0     13
16.0      8
17.0      4
18.0      4
26.0      3
19.0      3
23.0      2
27.0      1
24.0      1
21.0      1
20.0      1
22.0      1
Name: count, dtype: int64

In [202]:
dataini.nbda = dataini.nbda.fillna(0)
dataini

,modeEntree,modeSortie,duree,ghm2,dp,sexe,age,nbActe,nbRum,nbda,id,id_D,rea
0,8,9,0,01M37E,I671,2.0,76.0,4,1,0.0,l19,,0
1,8,8,3,01C061,I652,2.0,77.0,4,1,1.0,s7e,,0
2,8,7,13,01M303,I634,NaN,NaN,4,1,7.0,23f,,0
4,8,6,8,01M303,I635,1.0,71.0,4,1,9.0,otz,ld,1
6,8,7,8,01M302,I639,1.0,92.0,7,2,4.0,np7,,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1695,8,7,1,01M30T,I614,1.0,88.0,4,1,4.0,kjg,,0
1696,8,6,10,01M303,I635,1.0,81.0,10,3,7.0,gie,my,1
1697,8,8,8,01M301,I639,1.0,68.0,5,3,6.0,6bl,,0
1698,8,8,11,01M301,I676,2.0,28.0,16,5,7.0,7m8,,0


In [203]:
dataset = dataini[dataini['modeSortie'] != 9].drop(['id', 'id_D'], axis=1)
dataset

,modeEntree,modeSortie,duree,ghm2,dp,sexe,age,nbActe,nbRum,nbda,rea
1,8,8,3,01C061,I652,2.0,77.0,4,1,1.0,0
2,8,7,13,01M303,I634,NaN,NaN,4,1,7.0,0
4,8,6,8,01M303,I635,1.0,71.0,4,1,9.0,1
6,8,7,8,01M302,I639,1.0,92.0,7,2,4.0,0
7,8,8,8,01M301,I638,2.0,88.0,8,2,5.0,0
...,...,...,...,...,...,...,...,...,...,...,...
1695,8,7,1,01M30T,I614,1.0,88.0,4,1,4.0,0
1696,8,6,10,01M303,I635,1.0,81.0,10,3,7.0,1
1697,8,8,8,01M301,I639,1.0,68.0,5,3,6.0,0
1698,8,8,11,01M301,I676,2.0,28.0,16,5,7.0,0


In [204]:
OUT_PATH_S3 = BUCKET + '/dataset.parquet'

with fs.open(OUT_PATH_S3, mode='wb') as file_out:
    dataset.to_parquet(file_out, index=False, compression='snappy')

# 2. Feature Engineering

In [205]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, Normalizer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [206]:
OneHotEncoder(sparse_output=False, drop='first').fit_transform(SimpleImputer(strategy='most_frequent').fit_transform(dataset[['sexe']]))

array([[1.],
       [0.],
       [0.],
       ...,
       [0.],
       [1.],
       [0.]], shape=(1322, 1))

In [207]:
features = dataset.drop('rea', axis=1)
label = dataset['rea']

In [208]:
features.dtypes

modeEntree     object
modeSortie     object
duree           int32
ghm2           object
dp             object
sexe           object
age           float64
nbActe          int32
nbRum           int32
nbda          float64
dtype: object

In [209]:
num_features = features.select_dtypes(['int32', 'float64']).columns
cat_features = features.select_dtypes(['object']).columns

In [210]:
cat_transformer = Pipeline(steps =[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])

In [211]:
num_transformer = Pipeline(steps =[
    ('imputer', SimpleImputer()),
    ('scaler', StandardScaler())
])

In [212]:
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

In [213]:
preprocessor.fit_transform(features).toarray()

array([[-0.68288188,  0.37261423, -0.29461677, ...,  0.        ,
         0.        ,  1.        ],
       [ 0.35228593,  0.        , -0.29461677, ...,  0.        ,
         1.        ,  0.        ],
       [-0.16529798, -0.02521297, -0.29461677, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [-0.16529798, -0.22412657, -0.23119333, ...,  0.        ,
         1.        ,  0.        ],
       [ 0.14525237, -2.87630794,  0.46646456, ...,  0.        ,
         0.        ,  1.        ],
       [-0.78639866,  0.04109156, -0.54831055, ...,  0.        ,
         1.        ,  0.        ]], shape=(1322, 112))

In [214]:
#%whos

In [215]:
from sklearn.model_selection import train_test_split

In [216]:
X_train_val, X_test, y_train_val, y_test = train_test_split(features, label, random_state=18, test_size=0.1)

In [217]:
print(X_train_val.shape, X_test.shape)
print(y_train_val.shape, y_test.shape)

(1189, 10) (133, 10)
(1189,) (133,)


In [218]:
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, random_state=42, test_size=0.1)

## Logistic Regression

In [219]:
from sklearn.linear_model import LogisticRegression

In [220]:
lr = LogisticRegression()

In [221]:
pip_reg = Pipeline(steps=[
    ('preproc', preprocessor),
    ('classifier', lr)
])

In [222]:
pip_reg_fitted = pip_reg.fit(X_train, y_train)

In [223]:
pip_reg_fitted.predict_proba(X_val)[:,1]

array([0.05434258, 0.97548684, 0.98286929, 0.0211812 , 0.0631269 ,
       0.10254158, 0.02537978, 0.08273014, 0.02228209, 0.12270215,
       0.09570909, 0.02193838, 0.11686945, 0.12456897, 0.09138028,
       0.03862216, 0.97576334, 0.97922886, 0.02889055, 0.097124  ,
       0.08275629, 0.9772158 , 0.07117104, 0.16967249, 0.01482169,
       0.08644865, 0.02315506, 0.01568778, 0.08471653, 0.02734098,
       0.08538556, 0.10187251, 0.07345369, 0.05570633, 0.08937385,
       0.02476259, 0.08447695, 0.02633935, 0.08653747, 0.07501422,
       0.11059345, 0.98219666, 0.05484911, 0.08380927, 0.03944364,
       0.09274702, 0.97169355, 0.09488123, 0.11668844, 0.04474209,
       0.09788602, 0.13422667, 0.03167464, 0.16393663, 0.06383311,
       0.03766595, 0.02792632, 0.03207453, 0.06229111, 0.01839435,
       0.10002437, 0.93301576, 0.02332209, 0.1178995 , 0.02983813,
       0.10429969, 0.12366031, 0.06291443, 0.02252857, 0.16511388,
       0.09183082, 0.04045147, 0.07448081, 0.07143891, 0.97775

In [224]:
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

In [225]:
accuracy_score(y_val, pip_reg_fitted.predict(X_val))

0.9495798319327731

In [226]:
f1_score(y_val, pip_reg_fitted.predict(X_val))

0.8235294117647058

In [227]:
roc_auc_score(y_val, pip_reg_fitted.predict_proba(X_val)[:,1])

0.8207070707070707

## Random Forest

In [228]:
from sklearn.ensemble import RandomForestClassifier
from pprint import pprint
from sklearn.model_selection import GridSearchCV, RepeatedKFold, StratifiedKFold

In [229]:
rf=RandomForestClassifier(random_state=42)

In [230]:
pip_reg = Pipeline(steps=[
    ('preproc', preprocessor),
    ('classifier', rf)
])

In [231]:
param_rf = {
    'classifier__n_estimators' : [100,200,500],
    'classifier__max_depth':[10,20]
}

metric_grid = ['accuracy', 'f1', 'roc_auc']

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [232]:
grid_rf = GridSearchCV(estimator=pip_reg, param_grid=param_rf, cv=cv, scoring='roc_auc')

In [233]:
grid_rf_fitted = grid_rf.fit(X_train_val, y_train_val)
grid_rf_fitted.predict(X_val)

array([0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 1, 0, 0, 0, 0], dtype=int8)

In [234]:
from sklearn.svm import SVC

In [235]:
svm = SVC()

In [236]:
pip_svm=Pipeline(steps=[
    ('preproc', preprocessor),
    ('classifier', svm)
])

In [237]:
param_svm = {
    'classifier__kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'classifier__degree' : [2,3,4]
}

In [ ]:
models = [
    ('lr', lr, {'classifier__C' :[1,0]}),
    ('rf', rf, param_rf),
    ('svm', svm, param_svm)
]

preproc_list=[
    {'id' : 'basic', 'object' : preprocessor}
]

results=[]
for preproc in preproc_list:
    preproc_id = preproc['id'],
    preproc_object = preproc['object']

    for model_id, model_object, param in models:
        print(f'Modèle {model_id} with preprocessor {preproc_id}')
        pip=Pipeline(steps=[
            ('preproc', preproc_object),
            ('classifier', model_object)
        ])
        grid = GridSearchCV(
            estimator = pip,
            cv=cv,
            param_grid=param,
            scoring='roc_auc',
            refit=True
        ).fit(X_train_val, y_train_val)
        results.append(
            {'preprocessor' : preproc_id,
            'model' : model_id,
            'best_param' : grid.best_params_,
            'best_score' : grid.best_score_,
            'best_estimator' : grid.best_estimator_,
            'final_prediction' : grid.score(X_test, y_test)}
        )

Modèle lr with preprocessor ('basic',)
Modèle rf with preprocessor ('basic',)


/opt/python/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
5 fits failed out of a total of 10.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/python/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/python/lib/python3.13/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/opt/python/lib/python3.13/site-packages/sklearn/pipeline.py", line 663, in fit
    self._final_estimator.fit(Xt, y, **

Modèle svm with preprocessor ('basic',)


In [240]:
pd.DataFrame(results)

,preprocessor,model,best_param,best_score,final_prediction
0,"(basic,)",lr,{'classifier__C': 1},0.866766,0.850847
1,"(basic,)",rf,"{'classifier__max_depth': 10, 'classifier__n_e...",0.868047,0.907910
2,"(basic,)",svm,"{'classifier__degree': 2, 'classifier__kernel'...",0.847256,0.920904
